# HalluciGuard on Kaggle: blackboard vs. baselines on PopQA + FActScore bios

**Notebook settings:** Accelerator = **GPU T4 x2**, Internet = **on**. No API keys are needed:
the generator is Qwen2.5-7B-Instruct and every agent is an open model served locally with Ollama.

Order matters because the GPUs are shared:
1. **Stage 1**: Qwen generates the answers (uses both T4s, then exits and frees them).
2. **Stage 2**: one Ollama server **per GPU** (GPU0 -> port 11434, GPU1 -> port 11435), both holding Llama-3.1-8B + Mistral-7B;
   every agent call goes to the less busy GPU, so both T4s work in parallel. The blackboard and the baselines mitigate the answers.
3. **Stage 3**: scoring (the FActScore judge Gemma-2-9B gets both GPUs in its own phase) + comparison tables, zipped to `/kaggle/working/results.zip`.

Every step checkpoints to `runs/`; if the session dies, rerun the cells and they resume.

In [ ]:
# 0. code + dependencies
%cd /kaggle/working
!rm -rf HalluciGuard && git clone -q -b feature/real-blackboard https://github.com/Yaswanth8390/HalluciGuard.git
%cd /kaggle/working/HalluciGuard
!pip install -q -r requirements-kaggle.txt
!git log --oneline -3

In [ ]:
# 0b. offline tests (fake models, a few minutes). Optional, but catches environment problems early.
!python -m pytest tests -q
!python eval_selftest.py 2>&1 | grep -E "^ok|passed|Error|Traceback"

## Settings
Run once with `PILOT = True` (small, ~1 h) and check the smoke test + summaries. Then set `PILOT = False`
for the paper-size run (1,000 PopQA questions: a 95% CI of about ±3 points instead of ±5 at 300).
PopQA questions are split equally over the popularity terciles (head / torso / tail).

In [ ]:
PILOT = True           # True: quick end-to-end check. False: paper-size run (several hours)
N_POPQA, N_BIO = (60, 10) if PILOT else (999, 100)   # PopQA questions (multiple of 3), FActScore biographies
EXTRA_SEEDS = [] if PILOT else [1, 2]                # extra PopQA samples (300 each) for run-to-run variance
GEN2 = "microsoft/Phi-3.5-mini-instruct"             # second generator family (ungated); "" to skip
BUDGET_QA = 8          # LLM calls per PopQA question for blackboard / CoVe / CRAG
BUDGET_BIO = 24        # LLM calls per biography
WORKERS = 8            # parallel items = 2 GPUs x OLLAMA_NUM_PARALLEL (4)
PROBE = ""             # your trained probe .joblib; empty = auto-detect under /kaggle/input (add it as a Kaggle dataset)
N_DEV = 60 if PILOT else 300   # PopQA dev questions (disjoint from the eval set) for the probe threshold

import glob, json, os, time
if not PILOT and os.path.isdir("runs") and not os.path.exists("runs/.paper_run"):
    os.rename("runs", f"runs_pilot_{int(time.time())}")   # never mix pilot results into the paper run
os.makedirs("runs", exist_ok=True)
if not PILOT:
    open("runs/.paper_run", "w").close()
if not PROBE:
    found = sorted(glob.glob("/kaggle/input/**/*.joblib", recursive=True))
    PROBE = found[0] if found else ""
probe_arg = f"--probe {PROBE}" if PROBE else ""
GUARD, THRESH = "", 0.0
if PROBE:
    import joblib
    _b = joblib.load(PROBE)
    print("probe:", PROBE, {k: _b[k] for k in ("feature_dim", "n_train", "n_test", "test_metrics") if k in _b})
    assert _b.get("feature_dim") in (None, 3584), "expected a Qwen2.5-7B (d=3584) probe; check --layer / model"
    # the probe was trained on HaluEval questions and the eval is PopQA, so train/test overlap is impossible;
    # a bundle without train_questions (older train_probe.py) just cannot prove it, hence the flag
    GUARD = "" if _b.get("train_questions") else "--allow_train_overlap"
else:
    N_DEV = 0
    print("no probe found: every answer will be mitigated (--threshold 0)")

## 1. Data: PopQA (+ Wikipedia KB) and FActScore biographies

In [ ]:
def wiki_missing():
    """Eval questions whose Wikipedia article is not in the cache yet (fetch failed, e.g. rate limit)."""
    have = set()
    if os.path.exists("eval_data/wiki_cache.jsonl"):
        for l in open("eval_data/wiki_cache.jsonl", encoding="utf-8"):
            r = json.loads(l) if l.strip() else {}
            if r.get("text") or r.get("missing"):
                have.add(r["title"])
    files = ["eval_data/popqa.jsonl"] + [f"eval_data/popqa_s{s}.jsonl" for s in EXTRA_SEEDS]
    titles = {json.loads(l).get("s_wiki_title") for f in files if os.path.exists(f) for l in open(f, encoding="utf-8") if l.strip()}
    return sorted(t for t in titles if t and t not in have)

# Wikipedia rate-limits shared Kaggle IPs. Fetched articles are cached and failed ones are retried, so
# just repeat until nothing is missing (each round only fetches what is still missing).
for round_ in range(3):
    !python prepare_eval_data.py --dataset popqa --n {N_POPQA} --dev {N_DEV} --out eval_data/popqa.jsonl --wiki
    for s_ in EXTRA_SEEDS:
        !python prepare_eval_data.py --dataset popqa --n 300 --seed {s_} --out eval_data/popqa_s{s_}.jsonl --wiki
    missing_ = wiki_missing()
    print(f"articles still missing after round {round_ + 1}: {len(missing_)}", missing_[:10])
    if not missing_:
        break
    time.sleep(120)
!python prepare_factscore.py --n {N_BIO} --out eval_data/bio.jsonl
!wc -l eval_data/popqa.jsonl eval_data/popqa_kb.jsonl eval_data/bio.jsonl eval_data/bio_kb.jsonl

## 2. Stage 1: Qwen2.5-7B generates the answers (GPU)
PopQA uses the repo's short-answer system prompt (the model may answer `Unknown`). With your probe, Qwen's
layer-20 activations give every answer a `prob_hallucinated`. The probe threshold is then chosen on the
**dev split** (`choose_threshold.py`, max F1 for catching wrong answers), never on the eval questions.
Without a probe every answer is mitigated (`--threshold 0`).

In [ ]:
!python eval_pipeline.py --dataset eval_data/popqa.jsonl --generate_only --max_new_tokens 24 --batch_size 32 {probe_arg} --out_dir runs/popqa_gen
!python eval_bio.py --dataset eval_data/bio.jsonl --generate_only --batch_size 8 {probe_arg} --out_dir runs/bio
if PROBE:
    !python eval_pipeline.py --dataset eval_data/popqa_dev.jsonl --generate_only --max_new_tokens 24 --batch_size 32 {probe_arg} --out_dir runs/popqa_dev
for s_ in EXTRA_SEEDS:
    !python eval_pipeline.py --dataset eval_data/popqa_s{s_}.jsonl --generate_only --max_new_tokens 24 --batch_size 32 {probe_arg} --out_dir runs/popqa_gen_s{s_}
# second generator (different model family; no probe exists for it, so its runs mitigate every answer)
if GEN2:
    !python eval_pipeline.py --dataset eval_data/popqa.jsonl --generate_only --max_new_tokens 24 --batch_size 32 --model_id {GEN2} --out_dir runs/popqa_gen2
!nvidia-smi --query-gpu=memory.used --format=csv

In [ ]:
# probe threshold from the dev split (also prints the probe's dev AUROC on PopQA)
if PROBE:
    !python choose_threshold.py --dev runs/popqa_dev/generations.jsonl --out runs/popqa_dev/threshold.json
    THRESH = json.load(open("runs/popqa_dev/threshold.json"))["threshold"]
print("risk threshold used for the PopQA runs:", THRESH)

## 3. Stage 2: start the open agent models (Ollama) and map them to roles
Heterogeneous agents: verification and answering on **Llama-3.1-8B**, planning/writing on **Mistral-7B**,
the conflict resolver on a different family than the verifier, and a separate **FActScore judge (Gemma-2-9B)**
that is never an agent. None of them is Qwen, so they do not share the generator's errors.

**Two independent checks before an answer is accepted:** Llama verifies first, then the **Confirmer on
Mistral** (`LLM_VERIFIER2_MODEL`) must agree, using only passages about the question's entity. It always
checks answers read off the evidence, and checks Qwen's own answer when the probe's risk is >= 0.5. This is what
brings the hallucination rate down (the first pilot without it went from 23.7% to 30.0%).

**Both GPUs:** the script starts one server per T4 and loads both agent models on each (~12-13 GB per GPU).
`LLM_BASE_URL` lists both servers; each call goes to the less busy one, and a crashed server is skipped.
The status table at the end should show both models `100% GPU` on both ports.

In [ ]:
!bash scripts/kaggle_serve_agents.sh

In [ ]:
import os
AGENT_ENV = {
    "LLM_BASE_URL": "http://localhost:11434/v1,http://localhost:11435/v1",   # GPU0, GPU1
    "LLM_API_KEY": "ollama",
    "GROQ_MODEL": "llama3.1:8b",            # default for any role not listed below
    "LLM_JSON_MODE": "1",                   # ask the server for JSON objects (small models need it)
    "LLM_TIMEOUT": "300", "LLM_MAX_TOKENS": "1024",
    "LLM_VERIFIER_MODEL": "llama3.1:8b", "LLM_HYPOTHESIS_MODEL": "llama3.1:8b", "LLM_ANSWER_MODEL": "llama3.1:8b",
    "LLM_VERIFIER2_MODEL": "mistral:7b",    # Confirmer: independent second check by a different model family
    "LLM_DECOMPOSER_MODEL": "mistral:7b", "LLM_REFORMULATOR_MODEL": "mistral:7b", "LLM_COMPOSER_MODEL": "mistral:7b",
    "LLM_CORRECTOR_MODEL": "mistral:7b", "LLM_EXTRACTOR_MODEL": "mistral:7b", "LLM_ABSTENTION_MODEL": "mistral:7b",
    "LLM_CONFLICT_MODEL": "mistral:7b",     # resolver: different family than the verifier
    "LLM_BASELINE_MODEL": "llama3.1:8b",    # CoVe / CRAG use one model, as in their papers
    "LLM_JUDGE_MODEL": "gemma2:9b",         # FActScore judge, never an agent
}
os.environ.update(AGENT_ENV)
COMMON = (f"--answers_file runs/popqa_gen/generations.jsonl --kb_file eval_data/popqa_kb.jsonl --threshold {THRESH} {GUARD} {probe_arg} "
          f"--isolate_memory --no_llm_abstention --workers {WORKERS} --retries 3 --fallback_on_error")

### 3a. Live smoke test (10 questions)
Check the summary and a couple of traces before the long runs: are verdicts sensible, does JSON parse
(few `failed:` notes in the traces), are LLM calls within budget?

In [ ]:
!python eval_pipeline.py {COMMON} --limit 10 --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/smoke
!head -c 3000 runs/smoke/traces.jsonl
# the cross-model Confirmer must be active (Mistral re-checks what Llama supported)
cfg_ = json.load(open("runs/smoke/metrics.json"))["config"]
assert os.environ.get("LLM_VERIFIER2_MODEL") == "mistral:7b", "rerun the AGENT_ENV cell"
assert cfg_.get("confirm_risk") is not None and cfg_.get("grounding"), f"old code? config = {cfg_}"
n_conf_ = sum(json.dumps(json.loads(l)).count('"ks": "Confirmer"') for l in open("runs/smoke/traces.jsonl") if l.strip())
print(f"Confirmer cycles in the smoke test: {n_conf_} (confirm_risk={cfg_['confirm_risk']}, grounding={cfg_['grounding']})")
!bash scripts/kaggle_serve_agents.sh status   # both GPUs should be holding models and doing work

## 4. PopQA: all systems on the same answers
`pipeline` = the original fixed orchestrator, `cove` = Chain-of-Verification, `crag` = Corrective RAG,
`blackboard` = the new control loop. `--llm_only_control` (once) also grades Llama answering on its own.

With a probe this is **HalluciGuard as designed**: only answers the probe flags (score >= threshold) and
abstentions go to mitigation; the rest pass through untouched. Every system gets the same gate, so the
comparison is fair. Each `summary.md` also reports the probe's AUROC and precision/recall at the threshold
on these held-out PopQA answers (= how well a probe trained on HaluEval transfers to PopQA).

In [ ]:
for system in ["blackboard", "pipeline", "cove", "crag"]:
    extra = "--llm_only_control" if system == "blackboard" else ""
    !python eval_pipeline.py {COMMON} --system {system} --max_llm_calls {BUDGET_QA} {extra} --out_dir runs/popqa/{system}

In [ ]:
!python compare_systems.py --runs blackboard=runs/popqa/blackboard pipeline=runs/popqa/pipeline cove=runs/popqa/cove crag=runs/popqa/crag --reference pipeline --out runs/popqa/comparison.md
!cat runs/popqa/blackboard/summary.md

### 4a. Cost vs. accuracy (budget sweep) and error analysis
The blackboard at 4 / 8 / 16 LLM calls per question next to the other systems. "Same accuracy for fewer
calls" is a result in itself. `trace_report.py` sorts the blackboard's successes and failures into
categories (fixed by a competing answer, wrong answer kept, over-abstention, no evidence, ...) with examples.

In [ ]:
for b_ in [4, 16]:
    !python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {b_} --out_dir runs/popqa/bb_budget{b_}
!pip install -q matplotlib
!python plot_cost_accuracy.py --runs blackboard=runs/popqa/blackboard bb_budget4=runs/popqa/bb_budget4 bb_budget16=runs/popqa/bb_budget16 pipeline=runs/popqa/pipeline cove=runs/popqa/cove crag=runs/popqa/crag --out runs/popqa/cost_accuracy
!python trace_report.py --run runs/popqa/blackboard --examples 5

### 4b. What does the probe gate buy? (only with a probe)
Same blackboard, but every answer is mitigated (`--threshold 0`). Compare accuracy / hallucination and,
above all, **LLM calls**: the gate should keep most of the gain at a fraction of the cost, and it should
avoid "correct -> wrong" damage on answers that were fine.

In [ ]:
if PROBE:
    ALL = COMMON.replace(f"--threshold {THRESH}", "--threshold 0")
    !python eval_pipeline.py {ALL} --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/popqa/bb_mitigate_all
    !python compare_systems.py --runs probe_gated=runs/popqa/blackboard mitigate_all=runs/popqa/bb_mitigate_all --reference mitigate_all --out runs/popqa/probe_gate.md

## 5. Ablations (blackboard only, same budget)
Which knowledge sources carry the effect? Each run switches one off.

- `minus_confirmer`: no second model; one SUPPORTED verdict from the Verifier is enough (the setup of the
  first pilot, where hallucinations went up).
- `no_grounding`: passages about other entities may propose and confirm answers, and an answer need not
  be written in the evidence.
- `confirm_all` / `confirm_evidence_only`: the probe-scaled acceptance bar. By default a base answer
  needs the second model only when its probe risk is >= 0.5; these confirm every answer (`0`) or only
  evidence-derived answers (`1.01`). This shows what the probe buys inside the blackboard.

In [ ]:
for ks in ["Confirmer", "HypothesisGenerator", "QueryReformulator", "ConflictResolver"]:
    !python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {BUDGET_QA} --disable_ks {ks} --out_dir runs/popqa/bb_minus_{ks}
!python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {BUDGET_QA} --no_grounding --out_dir runs/popqa/bb_no_grounding
for name_, cr_ in [("confirm_all", 0), ("confirm_evidence_only", 1.01)]:
    !python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {BUDGET_QA} --confirm_risk {cr_} --out_dir runs/popqa/bb_{name_}
!python compare_systems.py --runs blackboard=runs/popqa/blackboard minus_confirmer=runs/popqa/bb_minus_Confirmer no_grounding=runs/popqa/bb_no_grounding confirm_all=runs/popqa/bb_confirm_all confirm_evidence_only=runs/popqa/bb_confirm_evidence_only minus_hypgen=runs/popqa/bb_minus_HypothesisGenerator minus_reform=runs/popqa/bb_minus_QueryReformulator minus_conflict=runs/popqa/bb_minus_ConflictResolver --reference blackboard --out runs/popqa/ablations.md

In [ ]:
# single-model ablation: every agent role on Llama-3.1-8B (does heterogeneity help?)
single = {k: "llama3.1:8b" for k in AGENT_ENV if k.startswith("LLM_") and k.endswith("_MODEL") and k != "LLM_JUDGE_MODEL"}
saved = {k: os.environ[k] for k in single}
os.environ.update(single)
!python eval_pipeline.py {COMMON} --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/popqa/bb_single_llama
os.environ.update(saved)
!python compare_systems.py --runs hetero=runs/popqa/blackboard single_llama=runs/popqa/bb_single_llama --reference single_llama --out runs/popqa/heterogeneity.md

## 5b. Second generator (a different model family)
The same systems on answers from `GEN2`, so the result is not specific to Qwen. There's no probe for this
model, so every answer is mitigated (`--threshold 0`). The agents (Llama / Mistral) never check their own
family's answers.

In [ ]:
if GEN2:
    COMMON2 = (f"--answers_file runs/popqa_gen2/generations.jsonl --kb_file eval_data/popqa_kb.jsonl --threshold 0 "
               f"--isolate_memory --no_llm_abstention --workers {WORKERS} --retries 3 --fallback_on_error")
    for system in ["blackboard", "pipeline", "cove", "crag"]:
        !python eval_pipeline.py {COMMON2} --system {system} --max_llm_calls {BUDGET_QA} --out_dir runs/popqa_gen2/{system}
    !python compare_systems.py --runs blackboard=runs/popqa_gen2/blackboard pipeline=runs/popqa_gen2/pipeline cove=runs/popqa_gen2/cove crag=runs/popqa_gen2/crag --reference pipeline --out runs/popqa_gen2/comparison.md

## 5c. Run-to-run variance (extra PopQA samples; paper run only)
The main blackboard and CRAG configurations on independently drawn PopQA samples. Report mean ± spread
next to the main table.

In [ ]:
import statistics
for s_ in EXTRA_SEEDS:
    C_ = COMMON.replace("runs/popqa_gen/generations.jsonl", f"runs/popqa_gen_s{s_}/generations.jsonl").replace(
         "eval_data/popqa_kb.jsonl", f"eval_data/popqa_s{s_}_kb.jsonl")
    for system in ["blackboard", "crag"]:
        !python eval_pipeline.py {C_} --system {system} --max_llm_calls {BUDGET_QA} --out_dir runs/popqa_s{s_}/{system}
if EXTRA_SEEDS:
    lines_ = ["| system | accuracy (mean ± sd) | hallucination (mean ± sd) | samples |", "|---|---|---|---|"]
    for system in ["blackboard", "crag"]:
        dirs = [f"runs/popqa/{system}"] + [f"runs/popqa_s{s_}/{system}" for s_ in EXTRA_SEEDS]
        ms = [json.load(open(f"{d}/metrics.json"))["with_blackboard"] for d in dirs if os.path.exists(f"{d}/metrics.json")]
        acc = [100 * m["accuracy"] for m in ms]; hal = [100 * m["hallucination_rate"] for m in ms]
        sd = lambda v: statistics.stdev(v) if len(v) > 1 else 0.0
        lines_.append(f"| {system} | {statistics.mean(acc):.1f} ± {sd(acc):.1f} | {statistics.mean(hal):.1f} ± {sd(hal):.1f} | {len(ms)} |")
    open("runs/popqa/variance.md", "w").write(chr(10).join(lines_) + chr(10))
    print(chr(10).join(lines_))

### Optional: knowledge-base coverage
Only half of the items have their gold passage in the KB: covered items should get corrected, uncovered ones
should abstain instead of hallucinating.

In [ ]:
!python eval_pipeline.py --answers_file runs/popqa_gen/generations.jsonl --kb_coverage 0.5 --threshold {THRESH} {GUARD} {probe_arg} --isolate_memory --no_llm_abstention --workers {WORKERS} --retries 3 --fallback_on_error --system blackboard --max_llm_calls {BUDGET_QA} --out_dir runs/popqa/bb_cov50

## 6. FActScore biographies (long-form): mitigation + scoring
Every biography is mitigated (`--threshold 0`): the probe was trained on one-word answers, so its score on a
300-word biography is out of distribution. The score is still recorded, and the summary reports how well it
predicts low FActScore (AUROC). That tells us whether the probe transfers to long-form text.

In [ ]:
# 6a. mitigation with the agents loaded (scoring skipped)
!python eval_bio.py --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --systems blackboard,pipeline,cove,crag --max_llm_calls {BUDGET_BIO} --workers {WORKERS} --out_dir runs/bio --skip_scoring

In [ ]:
# 6b. swap the agents for the judge on both GPUs, then score (mitigated rows are reused, not rerun)
!bash scripts/kaggle_serve_agents.sh judge
!python eval_bio.py --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --systems blackboard,pipeline,cove,crag --max_llm_calls {BUDGET_BIO} --workers {WORKERS} --out_dir runs/bio
!bash scripts/kaggle_serve_agents.sh agents   # back to the agents for any further runs

### 6c. Validate the FActScore judge (do this by hand after the run)
Exports 200 judged facts with their Wikipedia context and **hides the judge's label**. Download
`runs/bio/annotation/annotate.csv` (it's in results.zip), fill `human_label_1` (1 = the context supports the
fact, 0 = it does not), and ideally `human_label_2` by a second person. Then run, on any machine:
`python judge_agreement.py score --dir runs/bio/annotation`, which reports judge-vs-human kappa for the paper.

In [ ]:
!python judge_agreement.py export --bio_dir runs/bio --answers_file runs/bio/generations.jsonl --kb eval_data/bio_kb.jsonl --n 200 --out runs/bio/annotation

## 7. Results

In [ ]:
!cat runs/popqa/comparison.md; echo; cat runs/popqa/cost_accuracy.md; echo; cat runs/popqa/probe_gate.md 2>/dev/null; echo; cat runs/popqa/ablations.md; echo; cat runs/popqa_gen2/comparison.md 2>/dev/null; echo; cat runs/popqa/variance.md 2>/dev/null; echo; cat runs/bio/summary.md
!cd /kaggle/working/HalluciGuard && zip -qr /kaggle/working/results.zip runs eval_data/popqa*.jsonl eval_data/bio.jsonl -x "eval_data/*_kb.jsonl" && ls -lh /kaggle/working/results.zip